In [1]:
import re

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report

In [2]:
# ---- LOAD ----
# Structured sentence data
df = pd.read_csv("../data/generated_sentences.csv").dropna(subset=["sentence", "label"])

# Unstructured sentence data
# df = pd.read_csv("../data/filtered_sentences.csv").dropna(subset=["sentence", "label"])
df["label"] = df["label"].apply(lambda x: x.strip().split()[-1]) # TAKE LAST WORD AS LABEL
X_raw = df["sentence"].astype(str).tolist()
y_raw = df["label"].astype(str).tolist()

# ---- CHARACTER TOKENIZER ----
all_chars = sorted(set("".join(X_raw)))
char2idx = {c: i+2 for i, c in enumerate(all_chars)}
char2idx["<PAD>"] = 0
char2idx["<UNK>"] = 1

MAX_LEN = 50

In [3]:
def encode(sentence):
    ids = [char2idx.get(c, 1) for c in sentence]
    ids = ids[:MAX_LEN]
    ids += [0] * (MAX_LEN - len(ids))
    return ids

# ---- LABEL ENCODER ----
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
num_classes = len(le.classes_)
label_order = list(le.classes_)

X = [encode(s) for s in X_raw]
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# ---- SAMPLE WEIGHTS ----
def get_sample_weight(sentence, label, rule_boost=2.0):
    rules = {
        "PAST":     [r'[аэоөүию]н$', r'сан$|сэн$|сон$|сөн$'],
        "HABITUAL": [r'даг$|дэг$|дог$|дөг$'],
        "FUTURE":   [r'на$|нэ$|но$|нө$'],
    }
    for pattern in rules.get(label, []):
        if re.search(pattern, sentence.strip(), re.IGNORECASE):
            return rule_boost
    return 1.0

train_indices = list(range(len(X_train)))
train_weights = [
    get_sample_weight(X_raw[i], y_raw[i])
    for i in range(len(X_train))
]

# ---- DATASET ----
class SentenceDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float) if weights else torch.ones(len(X))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]

train_loader = DataLoader(SentenceDataset(X_train, y_train, train_weights), batch_size=16, shuffle=True)
test_loader  = DataLoader(SentenceDataset(X_test,  y_test),  batch_size=16)

# ---- TRANSFORMER MODEL ----
class TenseTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(MAX_LEN, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier  = nn.Linear(embed_dim, num_classes)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        positions    = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x            = self.embed(x) + self.pos_embed(positions)
        x            = self.dropout(x)
        padding_mask = (x.sum(dim=-1) == 0)
        x            = self.transformer(x, src_key_padding_mask=padding_mask)
        x            = x.mean(dim=1)
        return self.classifier(x)

model = TenseTransformer(
    vocab_size  = len(char2idx),
    embed_dim   = 64,
    num_heads   = 4,
    num_layers  = 2,
    num_classes = num_classes
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(reduction="none")

# ---- TRAIN ----
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0

    for X_batch, y_batch, w_batch in train_loader:
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss = (loss * w_batch).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y_batch).sum().item()

    acc = correct / len(X_train)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss:.4f} — Train Acc: {acc:.2%}")

Epoch 1/30 — Loss: 53.3882 — Train Acc: 34.64%
Epoch 2/30 — Loss: 50.3173 — Train Acc: 47.40%
Epoch 3/30 — Loss: 40.6553 — Train Acc: 55.73%
Epoch 4/30 — Loss: 36.6909 — Train Acc: 60.94%
Epoch 5/30 — Loss: 31.7616 — Train Acc: 70.83%
Epoch 6/30 — Loss: 28.2960 — Train Acc: 72.66%
Epoch 7/30 — Loss: 25.2100 — Train Acc: 77.08%
Epoch 8/30 — Loss: 25.5518 — Train Acc: 78.65%
Epoch 9/30 — Loss: 21.2455 — Train Acc: 81.25%
Epoch 10/30 — Loss: 21.2111 — Train Acc: 83.59%
Epoch 11/30 — Loss: 18.7750 — Train Acc: 84.90%
Epoch 12/30 — Loss: 16.8884 — Train Acc: 85.94%
Epoch 13/30 — Loss: 14.7879 — Train Acc: 88.28%
Epoch 14/30 — Loss: 16.0850 — Train Acc: 88.02%
Epoch 15/30 — Loss: 18.7594 — Train Acc: 83.59%
Epoch 16/30 — Loss: 13.5572 — Train Acc: 90.62%
Epoch 17/30 — Loss: 13.3026 — Train Acc: 90.10%
Epoch 18/30 — Loss: 16.2066 — Train Acc: 87.76%
Epoch 19/30 — Loss: 11.7263 — Train Acc: 90.10%
Epoch 20/30 — Loss: 11.0298 — Train Acc: 91.15%
Epoch 21/30 — Loss: 12.2442 — Train Acc: 90.62%
E

In [4]:
def run_seed(seed):
    """Full pipeline for a single random seed."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=seed
    )

    # Sample weights (must re-index against the new split)
    train_weights = [get_sample_weight(X_raw[i], y_raw[i]) for i in range(len(X_train))]

    train_loader = DataLoader(SentenceDataset(X_train, y_train, train_weights), batch_size=16, shuffle=True)
    test_loader  = DataLoader(SentenceDataset(X_test,  y_test),  batch_size=16)

    model = TenseTransformer(
        vocab_size  = len(char2idx),
        embed_dim   = 64,
        num_heads   = 4,
        num_layers  = 2,
        num_classes = num_classes
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(reduction="none")

    # Set ALL relevant seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)

    for epoch in range(EPOCHS):
        model.train()
        for X_batch, y_batch, w_batch in train_loader:
            optimizer.zero_grad()
            loss = (criterion(model(X_batch), y_batch) * w_batch).mean()
            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:
            all_preds.extend(model(X_batch).argmax(1).numpy())
            all_labels.extend(y_batch.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        "seed":      seed,
        "accuracy":  accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        "f1":        f1_score(all_labels, all_preds, average="weighted", zero_division=0),
    }


# ---- RUN ALL SEEDS ----
results = []
for seed in range(1, 11):
    print(f"Running seed {seed}/10...", end=" ", flush=True)
    row = run_seed(seed)
    results.append(row)
    print(f"Acc: {row['accuracy']:.2%}  P: {row['precision']:.2%}  F1: {row['f1']:.2%}")

# ---- SUMMARY TABLE ----
df = pd.DataFrame(results).set_index("seed")
print("\n", df.to_string(float_format="{:.4f}".format))
print("\n--- Averages ---")
print(df.mean().to_string(float_format="{:.4f}".format))

Running seed 1/10... Acc: 80.21%  P: 86.09%  F1: 80.85%
Running seed 2/10... 

/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Acc: 88.54%  P: 90.14%  F1: 88.62%
Running seed 3/10... Acc: 90.62%  P: 90.91%  F1: 90.67%
Running seed 4/10... Acc: 89.58%  P: 90.58%  F1: 89.42%
Running seed 5/10... Acc: 88.54%  P: 88.82%  F1: 88.58%
Running seed 6/10... Acc: 90.62%  P: 90.61%  F1: 90.60%
Running seed 7/10... Acc: 90.62%  P: 90.65%  F1: 90.59%
Running seed 8/10... Acc: 88.54%  P: 89.95%  F1: 88.48%
Running seed 9/10... Acc: 88.54%  P: 89.34%  F1: 88.56%
Running seed 10/10... Acc: 94.79%  P: 94.98%  F1: 94.83%

       accuracy  precision     f1
seed                            
1       0.8021     0.8609 0.8085
2       0.8854     0.9014 0.8862
3       0.9062     0.9091 0.9067
4       0.8958     0.9058 0.8942
5       0.8854     0.8882 0.8858
6       0.9062     0.9061 0.9060
7       0.9062     0.9065 0.9059
8       0.8854     0.8995 0.8848
9       0.8854     0.8934 0.8856
10      0.9479     0.9498 0.9483

--- Averages ---
accuracy    0.8906
precision   0.9021
f1          0.8912


In [5]:
# ---- RULE BIAS AT PREDICTION ----
def rule_bias(sentence, boost=10.0):
    bias = np.zeros(len(label_order))
    label_map = {l: i for i, l in enumerate(label_order)}
    rules = [
    (r'(?:сан|сэн|сон|сөн)$',         "PAST"),
    (r'(?:даг|дэг|дог|дөг)$',         "HABITUAL"),
    (r'(?:на|нэ|но|нө)$',             "FUTURE"),
    (r'[аэоөүию]н$',                  "PAST"),
        ]
    for pattern, label in rules:
        if re.search(pattern, sentence.strip(), re.IGNORECASE):
            if label in label_map:
                bias[label_map[label]] += boost
    return bias

# ---- PREDICT ----
def predict(sentences):
    model.eval()
    encoded = torch.tensor([encode(s) for s in sentences], dtype=torch.long)
    with torch.no_grad():
        logits = model(encoded).numpy()

    results = []
    for i, sentence in enumerate(sentences):
        final = logits[i] + rule_bias(sentence)
        pred  = np.argmax(final)
        results.append(le.inverse_transform([pred])[0])
    return results

In [6]:
# ---- TEST ----
sentences = [
    "Сайн ажилаас гардаг",
    "Тэр хүн хэн юм",
    "Хамгийн муу гэж харсан",
    "Англи бичиг сурна"
]

print("\nPredictions:\n")
for s, label in zip(sentences, predict(sentences)):
    print(f"  {s} -> {label}")


Predictions:

  Сайн ажилаас гардаг -> HABITUAL
  Тэр хүн хэн юм -> FUTURE
  Хамгийн муу гэж харсан -> PAST
  Англи бичиг сурна -> FUTURE
